In [3]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, 4ff49cd6-9471-48f5-a63a-1c8b3569d536, 5, Finished, Available, Finished, False)

##### <mark>**Monitoring / Logging**</mark>

**This is what makes the pipeline production-aware.**

**Why Monitoring / Logging matters**

In real projects, you must answer:

- Did the pipeline run?
- When did it start and end?
- Did it succeed or fail?
- How many rows were processed?
- What watermark was used?
- If it failed, where and why?

**Without logging, the pipeline is blind.**

**With logging, it becomes supportable and auditable.**

###### <mark>**Step 1: Create the logging table**</mark>

In [4]:
spark.sql("DROP TABLE IF EXISTS pipeline_run_log")

StatementMeta(, 4ff49cd6-9471-48f5-a63a-1c8b3569d536, 6, Finished, Available, Finished, False)

DataFrame[]

In [5]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType
from pyspark.sql import Row

log_schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("run_start_ts", TimestampType(), True),
    StructField("run_end_ts", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("rows_processed", LongType(), True),
    StructField("watermark_used", StringType(), True),
    StructField("remarks", StringType(), True)
])

empty_log_df = spark.createDataFrame([], schema=log_schema)

empty_log_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("pipeline_run_log")

StatementMeta(, 4ff49cd6-9471-48f5-a63a-1c8b3569d536, 7, Finished, Available, Finished, False)

###### <mark>**Step 2: Insert one sample success log**</mark>
**We’ll simulate logging for your timestamp incremental pipeline.**

In [6]:
from datetime import datetime
import uuid

pipeline_name = "nb_incremental_orders_watermark_timestamp"
run_id = str(uuid.uuid4())
run_start_ts = datetime.now()
run_end_ts = datetime.now()
status = "SUCCESS"
rows_processed = 3
watermark_used = "2026-03-07 09:00:00"
remarks = "Timestamp-based incremental load completed successfully"

log_data = [
    (
        pipeline_name,
        run_id,
        run_start_ts,
        run_end_ts,
        status,
        rows_processed,
        watermark_used,
        remarks
    )
]

log_df = spark.createDataFrame(log_data, schema=log_schema)

log_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("pipeline_run_log")

StatementMeta(, 4ff49cd6-9471-48f5-a63a-1c8b3569d536, 8, Finished, Available, Finished, False)

###### <mark>**Step 3: Verify logging table**</mark>

In [7]:
spark.sql("""
SELECT *
FROM pipeline_run_log
ORDER BY run_start_ts DESC
""").show(truncate=False)

StatementMeta(, 4ff49cd6-9471-48f5-a63a-1c8b3569d536, 9, Finished, Available, Finished, False)

+-----------------------------------------+------------------------------------+--------------------------+--------------------------+-------+--------------+-------------------+-------------------------------------------------------+
|pipeline_name                            |run_id                              |run_start_ts              |run_end_ts                |status |rows_processed|watermark_used     |remarks                                                |
+-----------------------------------------+------------------------------------+--------------------------+--------------------------+-------+--------------+-------------------+-------------------------------------------------------+
|nb_incremental_orders_watermark_timestamp|4f7096c8-13fb-4e63-9315-ef02a0ff0145|2026-03-30 02:20:34.275989|2026-03-30 02:20:34.276007|SUCCESS|3             |2026-03-07 09:00:00|Timestamp-based incremental load completed successfully|
+-----------------------------------------+---------------------

###### <mark>**Step 4 Failure logging simulation**</mark>

In [8]:
from datetime import datetime
import uuid

pipeline_name = "nb_incremental_orders_watermark_timestamp"
run_id = str(uuid.uuid4())
run_start_ts = datetime.now()
run_end_ts = datetime.now()
status = "FAILED"
rows_processed = 0
watermark_used = "2026-03-08 09:15:00"
remarks = "Simulated failure: source file missing / upstream dependency unavailable"

failure_log_data = [
    (
        pipeline_name,
        run_id,
        run_start_ts,
        run_end_ts,
        status,
        rows_processed,
        watermark_used,
        remarks
    )
]

failure_log_df = spark.createDataFrame(failure_log_data, schema=log_schema)

failure_log_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("pipeline_run_log")

StatementMeta(, 4ff49cd6-9471-48f5-a63a-1c8b3569d536, 10, Finished, Available, Finished, False)

<mark>**Step 5: Verify both success and failure logs**</mark>

In [1]:
spark.sql("""
SELECT pipeline_name, run_id, status, rows_processed, watermark_used, remarks
FROM pipeline_run_log
ORDER BY run_start_ts DESC
""").show(truncate=False)

StatementMeta(, 470d2bce-c39c-4902-ad52-faccdb56a40f, 3, Finished, Available, Finished, False)

+-----------------------------------------+------------------------------------+-------+--------------+-------------------+------------------------------------------------------------------------+
|pipeline_name                            |run_id                              |status |rows_processed|watermark_used     |remarks                                                                 |
+-----------------------------------------+------------------------------------+-------+--------------+-------------------+------------------------------------------------------------------------+
|nb_incremental_orders_watermark_timestamp|73c6472a-3537-45c6-9e0e-447085024cf1|FAILED |0             |2026-03-08 09:15:00|Simulated failure: source file missing / upstream dependency unavailable|
|nb_incremental_orders_watermark_timestamp|4f7096c8-13fb-4e63-9315-ef02a0ff0145|SUCCESS|3             |2026-03-07 09:00:00|Timestamp-based incremental load completed successfully                 |
+--------------